# Lab 07 - Retrieval-Qualität messen (Teil VII)

Sie implementieren die Kernmetriken selbst (Precision@k, Recall@k, MRR, MAP,
nDCG), prüfen sie gegen die Referenzbibliothek ranx und sehen, warum abgestufte
Relevanz und nDCG zusammengehören. Zum Schluss messen Sie, wie verlässlich das
Gold-Set selbst ist (Annotator-Übereinstimmung, Cohen's Kappa).

## 1. Setup: Lauf gegen das Gold-Set

Wir erzeugen mit dem Hybrid-Retriever für jede Gold-Query ein Ranking auf
Dokumentebene. Diese Runs werten wir anschließend mit eigenen Funktionen aus.

In [ ]:
import math

from common.corpus import load_corpus
from common.goldset import load_queries, qrels
from common import retrieval as R

docs = load_corpus()
index = R.HybridRetriever(R.chunk_corpus(docs))
queries = load_queries()
GOLD = qrels()   # {qid: {doc_id: grade}}

runs = {}
for q in queries:
    ranked = R.collapse_to_docs(index.search(q.text, k=10))
    runs[q.qid] = [doc for doc, _ in ranked]   # geordnete doc_id-Liste
print("Beispiel-Ranking q01:", runs["q01"][:5])
print("Gold zu q01:", GOLD["q01"])

## 2. Metriken von Hand

Binäre Relevanz für Precision und Recall (Grade >= 1 gilt als relevant),
abgestufte Relevanz für nDCG.

In [ ]:
def precision_at_k(ranking, gold, k):
    top = ranking[:k]
    return sum(1 for d in top if gold.get(d, 0) >= 1) / k


def recall_at_k(ranking, gold, k):
    rel_gesamt = sum(1 for g in gold.values() if g >= 1)
    if not rel_gesamt:
        return 0.0
    top = ranking[:k]
    return sum(1 for d in top if gold.get(d, 0) >= 1) / rel_gesamt


def reciprocal_rank(ranking, gold):
    for i, d in enumerate(ranking, start=1):
        if gold.get(d, 0) >= 1:
            return 1.0 / i
    return 0.0


def average_precision(ranking, gold):
    rel_gesamt = sum(1 for g in gold.values() if g >= 1)
    if not rel_gesamt:
        return 0.0
    treffer = 0
    summe = 0.0
    for i, d in enumerate(ranking, start=1):
        if gold.get(d, 0) >= 1:
            treffer += 1
            summe += treffer / i
    return summe / rel_gesamt


def dcg(grades, k):
    return sum((2 ** g - 1) / math.log2(i + 2) for i, g in enumerate(grades[:k]))


def ndcg_at_k(ranking, gold, k):
    grades = [gold.get(d, 0) for d in ranking]
    ideal = sorted(gold.values(), reverse=True)
    idcg = dcg(ideal, k)
    return dcg(grades, k) / idcg if idcg else 0.0

## 3. Eine Query per Hand nachvollziehen

Top-10 für eine Query mit Treffermarkierung. So sehen Sie, was die Zahlen
bedeuten, bevor Sie sie aggregieren.

In [ ]:
qid = "q08"   # Rücklauffilter-Intervall, mehrere relevante Docs
print(f"Query {qid}: {next(q.text for q in queries if q.qid == qid)}\n")
for rang, d in enumerate(runs[qid][:10], start=1):
    g = GOLD[qid].get(d, 0)
    mark = "x" if g >= 1 else " "
    print(f"  {rang:2d}. [{mark}] grade={g}  {d}")
print(f"\nP@5={precision_at_k(runs[qid], GOLD[qid], 5):.3f}  "
      f"R@5={recall_at_k(runs[qid], GOLD[qid], 5):.3f}  "
      f"RR={reciprocal_rank(runs[qid], GOLD[qid]):.3f}  "
      f"nDCG@5={ndcg_at_k(runs[qid], GOLD[qid], 5):.3f}")

## 4. Aggregat und Gegenprobe mit ranx

Wir mitteln über alle Queries und vergleichen mit ranx. Precision, Recall, MRR
und MAP stimmen exakt überein; bei nDCG bleibt eine winzige Abweichung durch
Implementierungsdetails (Tie-Handling, Truncation). Wer die Formel selbst
gebaut hat, versteht, was die Bibliothek rechnet, und vertraut ihr begründet.

In [ ]:
import pandas as pd
from ranx import Qrels, Run, evaluate

eigen = {
    "P@5":     sum(precision_at_k(runs[q], GOLD[q], 5) for q in GOLD) / len(GOLD),
    "Recall@5": sum(recall_at_k(runs[q], GOLD[q], 5) for q in GOLD) / len(GOLD),
    "MRR":     sum(reciprocal_rank(runs[q], GOLD[q]) for q in GOLD) / len(GOLD),
    "MAP":     sum(average_precision(runs[q], GOLD[q]) for q in GOLD) / len(GOLD),
    "nDCG@5":  sum(ndcg_at_k(runs[q], GOLD[q], 5) for q in GOLD) / len(GOLD),
}

ranx_run = Run({q: {d: (len(runs[q]) - i) for i, d in enumerate(runs[q])} for q in runs})
ranx_res = evaluate(Qrels(GOLD), ranx_run, ["precision@5", "recall@5", "mrr", "map", "ndcg@5"])

vergleich = pd.DataFrame({
    "eigen": [round(v, 3) for v in eigen.values()],
    "ranx":  [round(float(ranx_res[k]), 3) for k in
              ["precision@5", "recall@5", "mrr", "map", "ndcg@5"]],
}, index=list(eigen.keys()))
print(vergleich)

## 5. Warum nDCG abgestufte Relevanz braucht

Precision und Recall sehen nur "relevant ja/nein". nDCG belohnt, einen perfekten
Treffer (Grade 3) vor einen schwachen (Grade 1) zu setzen. Wir vergleichen die
Aussagekraft binär gegen abgestuft an einer Query mit gemischten Graden.

In [ ]:
qid = "q03"   # erstes Öl: ein Grade-3, ein Grade-2, ein Grade-1 Dokument
print("Gold-Grade:", GOLD[qid])
print("Ranking:   ", runs[qid][:4])
print(f"nDCG@3 (abgestuft):           {ndcg_at_k(runs[qid], GOLD[qid], 3):.3f}")
binär = {d: 1 for d, g in GOLD[qid].items() if g >= 1}
print(f"nDCG@3, wenn alles Grade 1 wäre: {ndcg_at_k(runs[qid], binär, 3):.3f}")

## 6. Wie verlässlich ist das Gold-Set?

Ein Gold-Set ist selbst eine Messung mit Fehler. Zwei Annotatoren stimmen nicht
perfekt überein. Wir simulieren einen zweiten Annotator (leicht abweichende
Urteile) und berechnen das quadratisch gewichtete Cohen's Kappa. Unter etwa
0,6 ist die Annotation zu verrauscht, um Systeme fein zu unterscheiden.

In [ ]:
from sklearn.metrics import cohen_kappa_score

# Annotator A = Gold; Annotator B weicht bei einigen Paaren um eine Stufe ab.
paare = [(q, d) for q in GOLD for d in GOLD[q]]
A = [GOLD[q][d] for q, d in paare]
B = list(A)
for i in range(0, len(B), 4):       # jedes vierte Urteil um eine Stufe senken
    B[i] = max(0, B[i] - 1)
kappa = cohen_kappa_score(A, B, weights="quadratic")
print(f"Annotierte Paare: {len(paare)}")
print(f"Quadratisch gewichtetes Cohen's Kappa (A vs B): {kappa:.3f}")

## Aufgabe 1: Precision@k gegen Recall@k

Berechnen Sie für k = 1, 3, 5, 10 den mittleren Precision@k und Recall@k.
Beschreiben Sie den Trade-off: was passiert mit beiden, wenn k wächst?

In [ ]:
# Ihre Lösung:

### Lösung

In [ ]:
for k in (1, 3, 5, 10):
    p = sum(precision_at_k(runs[q], GOLD[q], k) for q in GOLD) / len(GOLD)
    r = sum(recall_at_k(runs[q], GOLD[q], k) for q in GOLD) / len(GOLD)
    print(f"k={k:2d}: P@k={p:.3f}  R@k={r:.3f}")
print("-> Mit wachsendem k steigt der Recall, die Precision fällt tendenziell.")

## Aufgabe 2: Schwächste Query finden

Welche Query hat den niedrigsten nDCG@5? Schauen Sie sich ihr Ranking an und
überlegen Sie: ist das ein Retrieval-Problem (relevantes Dokument zu weit
unten) oder ein Gold-Problem (Urteil fragwürdig)?

In [ ]:
# Ihre Lösung:

### Lösung

In [ ]:
rang = sorted(queries, key=lambda q: ndcg_at_k(runs[q.qid], GOLD[q.qid], 5))
for q in rang[:3]:
    print(f"  nDCG@5={ndcg_at_k(runs[q.qid], GOLD[q.qid], 5):.3f}  {q.qid}  {q.text}")
    print("    Ranking:", runs[q.qid][:5], " Gold:", GOLD[q.qid])

## Diskussion

- nDCG und Recall sind hier Pflicht, MAP und Kappa nach Bedarf. Welche Metrik
  passt zu Ihrem Use Case: muss das eine beste Dokument oben stehen (MRR/nDCG)
  oder möglichst alle relevanten gefunden werden (Recall)?
- Ihre eigenen Metriken und ranx stimmen überein. Warum trotzdem eine
  etablierte Bibliothek in Produktion verwenden?
- Wenn Kappa niedrig ist: lohnt es sich überhaupt, zwei Systeme mit 0,01
  nDCG-Differenz zu vergleichen?